# TinyGPT-Opt on a Colab T4

Runtime → Change runtime type → **T4 GPU**, then run top to bottom.
The training runs take a while. Each one appends a row to `results/results.csv`.

In [ ]:
!nvidia-smi
!git clone https://github.com/KirtiK07/tinygpt-opt.git
%cd tinygpt-opt
!pip install -q tiktoken pyyaml

## Correctness tests
Every optimization must pass before its numbers count. On the GPU, the fp16 GradScaler test runs too.

In [ ]:
!pytest -q -rs

## Data
Tiny Shakespeare is a quick smoke test. TinyStories is the main dataset; `--max_train_docs` caps it for a faster run.

In [ ]:
!python -m tinygpt.data --dataset shakespeare
!python -m tinygpt.train --config configs/shakespeare.yaml

In [ ]:
!python -m tinygpt.data --dataset tinystories --max_train_docs 300000

## Training ladder (steps 0-4)
Same model, data, seed and step count on every rung.

In [ ]:
!python -m tinygpt.train --config configs/base.yaml

In [ ]:
!python -m tinygpt.train --config configs/amp.yaml

In [ ]:
!python -m tinygpt.train --config configs/fused.yaml

In [ ]:
!python -m tinygpt.train --config configs/ckpt.yaml

In [ ]:
!python -m tinygpt.train --config configs/compile.yaml

## Inference ladder (steps 5-7)

In [ ]:
!python -m tinygpt.profile infer --ckpt out/compile/ckpt.pt

## Results

In [ ]:
!python -m tinygpt.profile report
!python -m tinygpt.profile plot
from IPython.display import Image, display
display(Image('results/throughput.png'))
display(Image('results/peak_memory.png'))

In [ ]:
!python -m tinygpt.generate --ckpt out/compile/ckpt.pt --prompt 'Once upon a time' --num_samples 3

## fp16 NaN case study
The broken run is expected to stop with a non-finite loss (exit code 2).

In [ ]:
!python -m tinygpt.train --config configs/nan_repro.yaml
!python -m tinygpt.train --config configs/nan_fixed.yaml
!python -m tinygpt.profile nan_plot
display(Image('results/nan_loss.png'))
display(Image('results/nan_grad_norms.png'))

## Save results
Download `results/` and commit it to the repo, then fill in the README tables.

In [ ]:
!zip -r results.zip results out/*/metrics.csv out/*/sample.txt
from google.colab import files
files.download('results.zip')